<a href="https://colab.research.google.com/github/SafaaMahbub/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-25%20%E2%80%94%20Cleaning%20Gauntlet%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5
...,...,...,...,...,...
266,266,Cheeseburger,Merch,1.0,7.5
147,147,Cheeseburger,Merch,NaN,24
299,299,cheese burger,food,1.0,$12.00
58,58,rain poncho,RainGear,1.0,$7.50


### TODO 1 — drop duplicates

In [3]:
# TODO
clean = df.drop_duplicates().copy()
count_duplicates = df.duplicated().sum()
log('duplicate','dropping duplicated rows',count_duplicates)

[duplicate] dropping duplicated rows (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
clean['price'] =(clean['price']
                 .str.replace('$','',regex=False)
                 .str.strip()
                 .astype(float))
print('dtype now:', clean['price'].dtype)
log('price','normalize/standardize price column',len(clean['price']))

dtype now: float64
[price] normalize/standardize price column (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
missing_rows = clean['qty'].isna().sum()
negative_rows = (clean['qty'] < 0).sum()

clean = clean[clean['qty'].notna() & (clean['qty'] > 0)].copy()

log('qty','dropping negative qty rows',negative_rows)
log('qty','dropping missing qty rows',missing_rows)

[qty] dropping negative qty rows (13 row(s))
[qty] dropping missing qty rows (12 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
before_mapping = clean.copy()

In [7]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
before_mapping = clean.copy()
print(clean['item'].value_counts())
ITEM_MAP = {
    'Foam Finger' : 'foam finger',
    'Cheeseburger' : 'cheese burger',
    'Rain Poncho' : 'rain poncho',
}

print("BEFORE:")
print(before_mapping['item'].value_counts())

clean['item'] = clean['item'].replace(ITEM_MAP)

count = 0
for i in range(len(before_mapping['item'])):
  if(before_mapping['item'].iloc[i]!=clean['item'].iloc[i]):
    count=count+1

log('item','changing variants to an explicit map dictionary to make the item column unique', count)

print("AFTER:")
print(clean['item'].value_counts())

item
Foam Finger      61
cheese burger    55
Cheeseburger     54
Rain Poncho      52
rain poncho      49
foam finger      44
Name: count, dtype: int64
item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
BEFORE:
item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[item] changing variants to an explicit map dictionary to make the item column unique (149 row(s))
AFTER:
item
foam finger      97
rain poncho      91
cheese burger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [8]:
# TODO
before_category_mapping =clean.copy()
print(before_category_mapping['category'].value_counts())
CATEGORY_MAP = {
    'Apparel': 'merch',      # decision: apparel rolls up into merch
    'rain-gear': 'raingear',
    'RainGear': 'raingear',
    'Food' : 'food',
    'Merch': 'merch'
}
clean['category'] = clean['category'].replace(CATEGORY_MAP)
print()

count = 0
for i in range(len(before_category_mapping['category'])):
  if(before_category_mapping['category'].iloc[i]!=clean['category'].iloc[i]):
    count=count+1

print(clean['category'].value_counts())
log('category','standardizing the category column to cut variants of the same word', count)

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64

category
food        95
merch       94
raingear    86
Name: count, dtype: int64
[category] standardizing the category column to cut variants of the same word (231 row(s))


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [9]:
assert clean.duplicated().sum() == 0
assert clean['qty'].min() >= 1
assert clean['price'].dtype == float
# TODO: assert something about item
assert clean['item'].str.islower().all()
# TODO: assert something about category
assert clean['category'].str.islower().all()
print('clean:', df.shape)

clean: (315, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [10]:
# TODO
clean['revenue'] = clean['qty']*clean['price']
print(clean.groupby('category')['revenue'].sum().sort_values(ascending=False))


category
food        1656.0
merch       1572.0
raingear    1512.0
Name: revenue, dtype: float64


**What I would tell the vendor:** To increase the revenue, I would add maybe more food items to attract more customers to buy food. Maybe move the day to a rainy day so that more people are willing to buy rain gear

### TODO 8 — read back your log

In [11]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,duplicate,dropping duplicated rows,15
1,price,normalize/standardize price column,300
2,qty,dropping negative qty rows,13
3,qty,dropping missing qty rows,12
4,item,changing variants to an explicit map dictionar...,149
5,category,standardizing the category column to cut varia...,231


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

**a)** step 1 because it is impossible to read and compute the total revneue since hte price column was considered an object column. The revenue **before** was = 0 and the revenue **after** = 4740.00

**b)** step 3 by dropping the negative quantity rows. Other decisions could have been to just use the absolute values of qty to get the actuall gross revenue. I choose to just get the net revenue including revenue because it was important for me to know how much total revenue I could use to buy resources in the future. If the qty included the negative numbers, the total revenue would have decreased.